# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Mã nguồn hoàn chỉnh cho toàn bộ các thí nghiệm E0 → E4 và đánh giá theo chuẩn đề bài.

## 0. Môi trường (Colab T4)
Kiểm tra GPU T4 (16GB VRAM), cài đặt thư viện và cấu hình Kaggle API credentials từ Colab Secrets.

In [ ]:
# Tự động clone repo nếu đang chạy trên Google Colab
import os
if not os.path.exists('scripts') and os.path.exists('/content'):
    !git clone https://github.com/ChunHieu/K4-Track-4-Day10-VisionLanguageModel.git
    %cd K4-Track-4-Day10-VisionLanguageModel

# Kiểm tra GPU và cài đặt thư viện
!nvidia-smi -L
!pip -q install -r requirements-colab.txt
!python -m pytest -q

In [ ]:
import os
import shutil
from pathlib import Path

# Mount Google Drive để lưu adapter an toàn (tránh mất session Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BACKUP = '/content/drive/MyDrive/day10_adapters'
    os.makedirs(DRIVE_BACKUP, exist_ok=True)
    print('Mounted Google Drive thành công!')
except Exception as e:
    print(f'Không mount được Google Drive (bỏ qua nếu chạy ngoài Colab): {e}')
    DRIVE_BACKUP = None

# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print('Đã cấu hình Kaggle credentials từ Colab Secrets.')
except Exception as e:
    print('Lưu ý: Chưa lấy được Kaggle secret từ Colab. Hãy chắc chắn bạn đã cấu hình KAGGLE_USERNAME và KAGGLE_KEY trong Secrets.')

## 1. Dữ liệu: tải và chia split
Tải bộ dữ liệu hư hại mặt đường từ Kaggle và chia thành 3 tập train, val, test theo phiên quay (session).

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
import json
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from scripts.labcommon import read_jsonl, CLASSES

train_records = read_jsonl('data/splits/train.jsonl')
val_records = read_jsonl('data/splits/val.jsonl')
test_records = read_jsonl('data/splits/test.jsonl')

print(f'Số lượng mẫu: Train={len(train_records)}, Val={len(val_records)}, Test={len(test_records)}')

meta = json.loads(Path('data/splits/meta.json').read_text(encoding='utf-8'))
print(f'Dataset version: {meta["dataset_version"]}')
print('Phân bố số lượng box từng lớp:', meta['counts'])

# Trực quan hoá 4 ảnh mẫu từ tập train kèm nhãn YOLO
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for i, ax in enumerate(axes.flat):
    sample = train_records[i]
    img = Image.open(f'data/{sample["image_path"]}').convert('RGB')
    draw = ImageDraw.Draw(img)
    w, h = sample['width'], sample['height']
    for obj in sample['objects']:
        xc, yc, bw, bh = obj['yolo']
        x1, y1 = (xc - bw / 2) * w, (yc - bh / 2) * h
        x2, y2 = (xc + bw / 2) * w, (yc + bh / 2) * h
        draw.rectangle([x1, y1, x2, y2], outline='red', width=2)
        draw.text((x1, max(0, y1 - 12)), obj['label'], fill='yellow')
    ax.imshow(img)
    ax.set_title(f"{sample['image_id']} ({sample['session']}) - {len(sample['objects'])} boxes")
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000
Chuyển đổi nhãn từ YOLO [0, 1] sang định dạng JSON grounding [0, 1000] theo quy ước của Qwen3-VL.

In [ ]:
def yolo_to_box(label, yolo):
    """Chuyển [xc, yc, w, h] chuẩn hoá [0,1] sang [x1, y1, x2, y2] số nguyên [0, 1000]."""
    xc, yc, w, h = yolo
    x1 = int(round(max(0.0, min(1000.0, (xc - w / 2) * 1000))))
    y1 = int(round(max(0.0, min(1000.0, (yc - h / 2) * 1000))))
    x2 = int(round(max(0.0, min(1000.0, (xc + w / 2) * 1000))))
    y2 = int(round(max(0.0, min(1000.0, (yc + h / 2) * 1000))))
    if x2 <= x1:
        x2 = min(1000, x1 + 1)
    if y2 <= y1:
        y2 = min(1000, y1 + 1)
    return {'label': label, 'bbox_2d': [x1, y1, x2, y2]}

def objects_to_answer(objects):
    """Sắp xếp box trên->dưới, trái->phải và dump thành chuỗi JSON gọn."""
    boxes = [yolo_to_box(o['label'], o['yolo']) for o in objects]
    boxes.sort(key=lambda b: (b['bbox_2d'][1], b['bbox_2d'][0]))
    return json.dumps(boxes, separators=(',', ':'))

PROMPT = (
    'Locate all road damage in this image. '
    'Classes: pothole, crack, manhole. '
    'Output JSON list format: [{"label": "class_name", "bbox_2d": [x1, y1, x2, y2]}]. '
    'Coordinates must be integers normalized to 0-1000 scale. If none, return [].'
)

# Kiểm tra roundtrip trên 1 mẫu
sample = train_records[0]
answer_str = objects_to_answer(sample['objects'])
print('Sample JSON answer:', answer_str)
parsed_back = json.loads(answer_str)
print('Roundtrip parsing test passed! Number of boxes:', len(parsed_back))

## 3. E0: zero-shot
Nạp Qwen3-VL-4B-Instruct gốc ở chế độ 4-bit (NF4), giữ ViT và lm_head ở FP16. Suy luận trên tập test để lấy mốc baseline.

In [ ]:
import torch
from transformers import BitsAndBytesConfig, AutoProcessor
try:
    from transformers import Qwen3VLForConditionalGeneration
except ImportError:
    try:
        from transformers import AutoModelForImageTextToText as Qwen3VLForConditionalGeneration
    except ImportError:
        from transformers import AutoModelForVision2Seq as Qwen3VLForConditionalGeneration
from scripts.labcommon import write_jsonl
from tqdm import tqdm

MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'

# Cấu hình lượng tử hoá NF4 4-bit, giữ ViT và lm_head ở fp16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=['visual', 'lm_head']
)

print('Đang nạp processor và model Qwen3-VL-4B...')
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = 'left'

model_e0 = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={'': 0}
)
model_e0.eval()

# Suy luận trên toàn bộ 311 ảnh của test set
preds_e0 = []
print('Đang chạy suy luận E0 (zero-shot) trên test set...')
with torch.no_grad():
    for item in tqdm(test_records):
        img_path = Path('data') / item['image_path']
        image = Image.open(img_path).convert('RGB')
        messages = [
            {'role': 'user', 'content': [
                {'type': 'image', 'image': image},
                {'type': 'text', 'text': PROMPT}
            ]}
        ]
        inputs = processor.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors='pt'
        ).to('cuda')
        
        prompt_len = inputs['input_ids'].shape[1]
        output_ids = model_e0.generate(**inputs, max_new_tokens=256, do_sample=False)
        raw_text = processor.decode(output_ids[0, prompt_len:], skip_special_tokens=True).strip()
        preds_e0.append({'image_id': item['image_id'], 'raw': raw_text})

os.makedirs('work', exist_ok=True)
write_jsonl('work/e0.jsonl', preds_e0)
print('Đã ghi dự đoán E0 vào work/e0.jsonl')

# Giải phóng bộ nhớ model E0
del model_e0
torch.cuda.empty_cache()

In [ ]:
# Chấm điểm E0
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

e0_res = json.loads(Path('report/evidence/e0.json').read_text(encoding='utf-8'))
print(f"\nE0 Parse valid rate: {e0_res['parse_valid_rate']:.3f}, F1: {e0_res['micro']['f1']:.3f}")
print('\n--- 3 Output thô đầu tiên của E0 ---')
for p in preds_e0[:3]:
    print(f"[{p['image_id']}]: {p['raw']}")

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss
Xây dựng PyTorch Dataset và Collate Function với Loss Masking: chỉ tính loss trên assistant response (token câu trả lời), mask nhãn = -100 trên toàn bộ prompt và 220 visual tokens, đồng thời hỗ trợ mm_token_type_ids cho Multimodal RoPE.

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch

IMAGE_PAD_ID = processor.tokenizer.convert_tokens_to_ids('<|image_pad|>')

class RoadDamageDataset(Dataset):
    def __init__(self, records, processor, prompt):
        self.records = records
        self.processor = processor
        self.prompt = prompt

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        item = self.records[idx]
        image = Image.open(Path('data') / item['image_path']).convert('RGB')
        answer = objects_to_answer(item['objects'])
        
        user_content = [
            {'type': 'image', 'image': image},
            {'type': 'text', 'text': self.prompt}
        ]
        user_messages = [{'role': 'user', 'content': user_content}]
        full_messages = [
            {'role': 'user', 'content': user_content},
            {'role': 'assistant', 'content': [{'type': 'text', 'text': answer}]}
        ]
        
        prompt_enc = self.processor.apply_chat_template(
            user_messages, add_generation_prompt=True,
            tokenize=True, return_dict=True, return_tensors='pt'
        )
        full_enc = self.processor.apply_chat_template(
            full_messages, add_generation_prompt=False,
            tokenize=True, return_dict=True, return_tensors='pt'
        )
        
        prompt_len = prompt_enc['input_ids'].shape[1]
        input_ids = full_enc['input_ids'][0]
        
        # MASK LOSS: Gán nhãn -100 cho toàn bộ prompt và visual tokens
        labels = input_ids.clone()
        labels[:prompt_len] = -100
        
        # Tự động tạo mm_token_type_ids: 1 cho token ảnh, 0 cho token chữ
        if 'mm_token_type_ids' in full_enc and full_enc['mm_token_type_ids'] is not None:
            mm_token_type_ids = full_enc['mm_token_type_ids'][0]
        else:
            mm_token_type_ids = (input_ids == IMAGE_PAD_ID).to(torch.int64)
        
        return {
            'input_ids': input_ids,
            'labels': labels,
            'mm_token_type_ids': mm_token_type_ids,
            'pixel_values': full_enc.get('pixel_values'),
            'image_grid_thw': full_enc.get('image_grid_thw'),
            'prompt_len': prompt_len
        }

def collate_fn(batch):
    pad_id = processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id
    max_len = max(x['input_ids'].shape[0] for x in batch)
    
    input_ids_list, labels_list, attention_mask_list = [], [], []
    mm_type_list = []
    
    for x in batch:
        cur_len = x['input_ids'].shape[0]
        pad_size = max_len - cur_len
        
        in_ids = torch.cat([x['input_ids'], torch.full((pad_size,), pad_id, dtype=torch.long)])
        lbls = torch.cat([x['labels'], torch.full((pad_size,), -100, dtype=torch.long)])
        att_mask = torch.cat([torch.ones(cur_len, dtype=torch.long), torch.zeros(pad_size, dtype=torch.long)])
        mm_ids = torch.cat([x['mm_token_type_ids'], torch.zeros((pad_size,), dtype=torch.long)])
        
        input_ids_list.append(in_ids)
        labels_list.append(lbls)
        attention_mask_list.append(att_mask)
        mm_type_list.append(mm_ids)
        
    pixel_values = torch.cat([x['pixel_values'] for x in batch if x['pixel_values'] is not None], dim=0)
    image_grid_thw = torch.cat([x['image_grid_thw'] for x in batch if x['image_grid_thw'] is not None], dim=0)
    
    return {
        'input_ids': torch.stack(input_ids_list),
        'attention_mask': torch.stack(attention_mask_list),
        'labels': torch.stack(labels_list),
        'mm_token_type_ids': torch.stack(mm_type_list),
        'pixel_values': pixel_values,
        'image_grid_thw': image_grid_thw
    }

# Khởi tạo dataset và DataLoader chung (batch_size=2)
train_dataset = RoadDamageDataset(train_records, processor, PROMPT)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
print('Đã cập nhật xong Dataset và DataLoader! Sẵn sàng huấn luyện.')

## 5. Setup LoRA / đóng băng cho E1, E2, E3
Hàm `apply_setup(model, setup_name)` đóng băng toàn bộ mô hình và chỉ mở các khối mục tiêu:
- E1: LoRA trên LLM (36 layers × 7 modules = 252 modules).
- E2: LoRA trên LLM + mở full Projector (`visual.merger`, `visual.deepstack_merger_list`) ở kiểu dữ liệu `float32`.
- E3: E2 + LoRA trên ViT (24 blocks × 4 modules = 96 modules). Tổng cộng 348 LoRA modules.

In [ ]:
from peft import LoraConfig, get_peft_model
import re

def apply_setup(model, setup_name, lora_r=16, lora_alpha=32):
    for p in model.parameters():
        p.requires_grad = False
    model.enable_input_require_grads()
    
    llm_pattern = r'.*language_model.*(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)'
    vit_pattern = r'.*visual\.blocks\.\d+\.(attn\.(qkv|proj)|mlp\.(linear_fc1|linear_fc2))'
    
    if setup_name in ['E1', 'E2'] or 'e4' in setup_name.lower():
        target_regex = llm_pattern
    elif setup_name == 'E3':
        target_regex = f'({llm_pattern}|{vit_pattern})'
    else:
        target_regex = llm_pattern
        
    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=0.05,
        target_modules=target_regex,
        bias='none',
        task_type='CAUSAL_LM'
    )
    
    peft_model = get_peft_model(model, lora_config)
    matched_lora = [n for n, m in peft_model.named_modules() if hasattr(m, 'lora_A')]
    print(f'[{setup_name}] Số lượng module gắn LoRA: {len(matched_lora)}')
    
    if setup_name in ['E2', 'E3']:
        proj_params_count = 0
        for name, param in peft_model.named_parameters():
            if 'visual.merger' in name or 'visual.deepstack_merger_list' in name:
                param.data = param.data.to(torch.float32)
                param.requires_grad = True
                proj_params_count += param.numel()
        print(f'[{setup_name}] Đã mở Projector ở float32: {proj_params_count:,} tham số (~109M)')
        
    trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    print(f'[{setup_name}] Tổng tham số trainable: {trainable_params:,}')
    return peft_model

## 6. Vòng huấn luyện
Vòng lặp huấn luyện chuẩn 300 steps tối ưu (AdamW lr=2e-4, cosine scheduler warmup 5%, gradient accumulation=4, fp16 autocast).
Lưu adapter LoRA và LƯU RIÊNG projector weights cho E2 và E3.

In [ ]:
import time
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

def train_setup(setup_name, peft_model, dataloader, max_steps=300, lr=2e-4):
    start_time = time.time()
    torch.cuda.reset_peak_memory_stats()
    
    trainable_p = [p for p in peft_model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable_p, lr=lr, weight_decay=0.0)
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=15, num_training_steps=max_steps)
    scaler = torch.amp.GradScaler('cuda')
    
    peft_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    peft_model.config.use_cache = False
    peft_model.train()
    
    step = 0
    accum_steps = 4
    data_iter = iter(dataloader)
    
    print(f'\n===== BẮT ĐẦU HUẤN LUYỆN {setup_name} ({max_steps} steps) =====')
    optimizer.zero_grad()
    
    while step < max_steps:
        accum_loss = 0.0
        for _ in range(accum_steps):
            try:
                batch = next(data_iter)
            except StopIteration:
                data_iter = iter(dataloader)
                batch = next(data_iter)
                
            batch = {k: v.to('cuda') if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            
            with torch.amp.autocast('cuda', dtype=torch.float16):
                outputs = peft_model(**batch)
                loss = outputs.loss / accum_steps
                
            if not torch.isfinite(loss):
                raise RuntimeError(f'Loss bị tràn số (NaN/Inf) tại step {step}! Hãy kiểm tra lr hoặc mask loss.')
                
            scaler.scale(loss).backward()
            accum_loss += loss.item()
            
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_p, 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad()
        
        step += 1
        if step % 50 == 0 or step == max_steps:
            current_loss = accum_loss * accum_steps
            print(f'Step [{step:3d}/{max_steps}] | Loss: {current_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}')
            
    train_minutes = (time.time() - start_time) / 60.0
    peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
    trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    
    # 1. Lưu adapter LoRA
    save_dir = f'work/adapters_{setup_name}'
    os.makedirs(save_dir, exist_ok=True)
    peft_model.save_pretrained(save_dir)
    
    # 2. LƯU RIÊNG PROJECTOR WEIGHTS CHO E2, E3
    if setup_name in ['E2', 'E3']:
        proj_weights = {
            k: v.cpu() for k, v in peft_model.named_parameters()
            if ('visual.merger' in k or 'visual.deepstack_merger_list' in k) and v.requires_grad
        }
        proj_file = os.path.join(save_dir, 'projector.pt')
        torch.save(proj_weights, proj_file)
        print(f'Đã lưu riêng Projector weights vào {proj_file}')
        
    if DRIVE_BACKUP:
        try:
            drive_dest = os.path.join(DRIVE_BACKUP, setup_name)
            shutil.copytree(save_dir, drive_dest, dirs_exist_ok=True)
            print(f'Đã sao lưu adapter lên Google Drive: {drive_dest}')
        except Exception as e:
            print(f'Lỗi backup drive: {e}')
            
    meta_info = {
        'trainable_params': int(trainable_params),
        'peak_vram_gb': round(float(peak_vram_gb), 2),
        'train_minutes': round(float(train_minutes), 2)
    }
    print(f'Hoàn thành {setup_name}: {meta_info}')
    return meta_info

## 7. Suy luận và chấm E1–E3
Nạp lại mô hình cùng adapter và projector weights đã lưu, chạy suy luận trên tập test và chấm điểm bằng `scripts/score.py` với metadata đầy đủ.

In [ ]:
import os
import json
import torch
import subprocess
from pathlib import Path
from tqdm import tqdm
from PIL import Image
from peft import PeftModel
from scripts.labcommon import write_jsonl

def evaluate_setup(setup_name, meta_info):
    print(f'\n--- BẮT ĐẦU ĐÁNH GIÁ {setup_name} TRÊN TEST SET ---')
    adapter_dir = f'work/adapters_{setup_name}'
    
    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, torch_dtype=torch.float16, device_map={'': 0}
    )
    model = PeftModel.from_pretrained(base_model, adapter_dir)
    
    if setup_name in ['E2', 'E3']:
        proj_file = os.path.join(adapter_dir, 'projector.pt')
        if os.path.exists(proj_file):
            proj_weights = torch.load(proj_file, map_location='cuda', weights_only=True)
            model.load_state_dict(proj_weights, strict=False)
            print(f'-> Đã nạp lại {len(proj_weights)} projector tensors thành công!')
        else:
            print(f'CẢNH BÁO: Không tìm thấy {proj_file}! Projector sẽ dùng trọng số gốc!')
            
    model.eval()
    preds = []
    with torch.no_grad():
        for item in tqdm(test_records, desc=f'Inference {setup_name}'):
            img_path = Path('data') / item['image_path']
            image = Image.open(img_path).convert('RGB')
            messages = [
                {'role': 'user', 'content': [
                    {'type': 'image', 'image': image},
                    {'type': 'text', 'text': PROMPT}
                ]}
            ]
            inputs = processor.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=True,
                return_dict=True, return_tensors='pt'
            ).to('cuda')
            
            prompt_len = inputs['input_ids'].shape[1]
            out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
            raw = processor.decode(out[0, prompt_len:], skip_special_tokens=True).strip()
            preds.append({'image_id': item['image_id'], 'raw': raw})
            
    os.makedirs('work', exist_ok=True)
    os.makedirs('report/evidence', exist_ok=True)
    pred_path = f'work/{setup_name.lower()}.jsonl'
    write_jsonl(pred_path, preds)
    
    out_json = f'report/evidence/{setup_name.lower()}.json'
    score_cmd = [
        'python', 'scripts/score.py',
        '--pred', pred_path,
        '--name', setup_name,
        '--out', out_json,
        '--meta', f'trainable_params={meta_info["trainable_params"]}',
        '--meta', f'peak_vram_gb={meta_info["peak_vram_gb"]}',
        '--meta', f'train_minutes={meta_info["train_minutes"]}'
    ]
    subprocess.run(score_cmd, check=True)
    
    del model, base_model
    torch.cuda.empty_cache()
    
    res = json.loads(Path(out_json).read_text(encoding='utf-8'))
    print(f'-> Hoàn tất chấm {setup_name}: F1={res["micro"]["f1"]:.3f}, CI95={res["f1_ci95"]}')
    return res

In [ ]:
# === THÍ NGHIỆM E1 ===
base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, torch_dtype=torch.float16, device_map={'': 0}
)
peft_e1 = apply_setup(base_model, 'E1')
meta_e1 = train_setup('E1', peft_e1, train_loader, max_steps=300)
del peft_e1, base_model
torch.cuda.empty_cache()

res_e1 = evaluate_setup('E1', meta_e1)

In [ ]:
# === THÍ NGHIỆM E2 ===
base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, torch_dtype=torch.float16, device_map={'': 0}
)
peft_e2 = apply_setup(base_model, 'E2')
meta_e2 = train_setup('E2', peft_e2, train_loader, max_steps=300)
del peft_e2, base_model
torch.cuda.empty_cache()

res_e2 = evaluate_setup('E2', meta_e2)

In [ ]:
# === THÍ NGHIỆM E3 ===
base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, torch_dtype=torch.float16, device_map={'': 0}
)
peft_e3 = apply_setup(base_model, 'E3')
meta_e3 = train_setup('E3', peft_e3, train_loader, max_steps=300)
del peft_e3, base_model
torch.cuda.empty_cache()

res_e3 = evaluate_setup('E3', meta_e3)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)
Thực hiện ablation study E4: Chọn biến `rank` (r=4 vs r=64) trên setup tốt nhất (thường là E1 hoặc E2).
Tạo trực quan hoá ≥ 6 ảnh bounding box minh hoạ cho báo cáo và chạy kiểm tra bài nộp bằng `check_submission.py`.

In [ ]:
# === THÍ NGHIỆM E4: RANK ABLATION STUDY ===
print('=== CHẠY E4: RANK ABLATION STUDY ===')

# 1. E4 với Rank = 4 (alpha = 8)
base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, torch_dtype=torch.float16, device_map={'': 0}
)
peft_e4_r4 = apply_setup(base_model, 'E1', lora_r=4, lora_alpha=8)
meta_e4_r4 = train_setup('e4_rank_4', peft_e4_r4, train_loader, max_steps=300)
del peft_e4_r4, base_model
torch.cuda.empty_cache()

res_e4_r4 = evaluate_setup('e4_rank_4', meta_e4_r4)

# 2. E4 với Rank = 64 (alpha = 128)
base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, torch_dtype=torch.float16, device_map={'': 0}
)
peft_e4_r64 = apply_setup(base_model, 'E1', lora_r=64, lora_alpha=128)
meta_e4_r64 = train_setup('e4_rank_64', peft_e4_r64, train_loader, max_steps=300)
del peft_e4_r64, base_model
torch.cuda.empty_cache()

res_e4_r64 = evaluate_setup('e4_rank_64', meta_e4_r64)

In [ ]:
# === TRỰC QUAN HOÁ ≥ 6 ẢNH DỰ ĐOÁN (RUBRIC B) ===
from scripts.score import parse_output, iou_xyxy, _gt_boxes

os.makedirs('report', exist_ok=True)
best_preds = {row['image_id']: row['raw'] for row in read_jsonl('work/e1.jsonl')}

tp_count, fp_fn_count = 0, 0
saved_images = []

for item in test_records:
    raw = best_preds.get(item['image_id'])
    parsed = parse_output(raw)
    if parsed is None:
        continue
    gt = _gt_boxes(item)
    
    img = Image.open(Path('data') / item['image_path']).convert('RGB')
    draw = ImageDraw.Draw(img)
    w, h = item['width'], item['height']
    
    has_match, has_mismatch = False, False
    for g in gt:
        x1, y1, x2, y2 = [g['bbox'][0]*w/1000, g['bbox'][1]*h/1000, g['bbox'][2]*w/1000, g['bbox'][3]*h/1000]
        draw.rectangle([x1, y1, x2, y2], outline='green', width=3)
        draw.text((x1, max(0, y1-12)), f"GT:{g['label']}", fill='green')
        
    for p in parsed:
        x1, y1, x2, y2 = [p['bbox'][0]*w/1000, p['bbox'][1]*h/1000, p['bbox'][2]*w/1000, p['bbox'][3]*h/1000]
        matched = any(iou_xyxy(p['bbox'], g['bbox']) >= 0.5 and p['label'] == g['label'] for g in gt)
        color = 'cyan' if matched else 'red'
        draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
        draw.text((x1, min(h-14, y2)), f"PRED:{p['label']}", fill=color)
        if matched:
            has_match = True
        else:
            has_mismatch = True
            
    if has_match and not has_mismatch and tp_count < 3:
        tp_count += 1
        save_file = f'report/vis_tp_{tp_count}.png'
        img.save(save_file)
        saved_images.append(save_file)
    elif has_mismatch and fp_fn_count < 3:
        fp_fn_count += 1
        save_file = f'report/vis_err_{fp_fn_count}.png'
        img.save(save_file)
        saved_images.append(save_file)
        
    if tp_count >= 3 and fp_fn_count >= 3:
        break

print(f'Đã lưu thành công {len(saved_images)} ảnh minh hoạ vào report/:')
for p in saved_images:
    print(' -', p)

In [ ]:
# Kiểm tra tính hợp lệ của bài nộp trước khi nộp
!python scripts/check_submission.py